# Graph Feature Construction

Takes formatted_transactions.csv and converts it into PyTorch Geometric tensors:
- x: node feature matrix (placeholder all-1s, no account profile data exists)
- edge_index: directed transaction graph (from_id to to_id)
- edge_attr: base numerical edge features per transaction
- timestamps: per-transaction relative elapsed seconds
- y: ground truth laundering labels (0 = clean, 1 = laundering)

1. Load the Formatted CSV

In [2]:
import pandas as pd
import numpy as np
import torch
import os

csv_path = os.path.join(os.path.dirname(os.path.abspath('.')), "model", "formatted_transactions.csv")
df_edges = pd.read_csv(csv_path)

print(f"Loaded {len(df_edges)} transactions")
print(f"Columns : {df_edges.columns.tolist()}")
df_edges.head(3)

Loaded 15000000 transactions
Columns : ['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,512274,382542,382542,19810,21.33,1,21.33,1,0,0
1,899651,672087,672087,19810,285062.94,4,285062.94,4,0,0
2,1108640,813025,827655,19810,5933.71,8,5933.71,8,2,0


2. Re-normalise Timestamp

Shifts timestamps so the earliest transaction starts at t=0.

In [3]:
df_edges["Timestamp"] = df_edges["Timestamp"] - df_edges["Timestamp"].min()

n_days = int(df_edges['Timestamp'].max() / (3600 * 24) + 1)
n_samples = len(df_edges)

timestamp_range = df_edges["Timestamp"].max() - 0
print(f'Timestamp range : {timestamp_range} seconds')
print(f'Dataset spans : {n_days} calendar days')
print(f'Total samples : {n_samples} transactions')

Timestamp range : 6430860 seconds
Dataset spans : 75 calendar days
Total samples : 15000000 transactions


3. Build Node Feature Matrix (x)

Nodes = unique bank accounts (all from_id and to_id values combined).
Since raw transaction logs contain no account profile metadata, every node gets a placeholder feature of 1.0.
The GNN will learn expressive node embeddings through message passing during training.

In [4]:
max_n_id = int(df_edges[['from_id', 'to_id']].to_numpy().max()) + 1

df_nodes = pd.DataFrame({
    'NodeID'  : np.arange(max_n_id),
    'Feature' : np.ones(max_n_id)
})

x = torch.tensor(df_nodes[['Feature']].to_numpy()).float()

print(f'Total unique account nodes : {max_n_id}')
print(f'Node feature tensor shape (num_nodes x num_node_features) : {tuple(x.shape)}')
print(f'All node feature values (all 1.0 placeholders) : {x.unique().tolist()}')

Total unique account nodes : 2061626
Node feature tensor shape (num_nodes x num_node_features) : (2061626, 1)
All node feature values (all 1.0 placeholders) : [1.0]


4. Build Edge Index (edge_index)

Represents the directed transaction graph as a 2 x num_edges integer tensor.
Row 0 = source account IDs (senders / from_id)
Row 1 = destination account IDs (receivers / to_id)

In [5]:
edge_index = torch.LongTensor(df_edges[['from_id', 'to_id']].to_numpy().T)

print(f'Edge index shape (2 x num_transactions) : {tuple(edge_index.shape)}')
print('Sample edges (from_id -> to_id) : ')
for i in range(3):
    print(f' Transaction {i} : Account {edge_index[0, i].item()} -> Account {edge_index[1, i].item()}')

Edge index shape (2 x num_transactions) : (2, 15000000)
Sample edges (from_id -> to_id) : 
 Transaction 0 : Account 382542 -> Account 382542
 Transaction 1 : Account 672087 -> Account 672087
 Transaction 2 : Account 813025 -> Account 827655


5. Build Base Edge Feature Matrix (edge_attr)

4 numerical features per transaction:
1. Timestamp - when the transaction happened (relative seconds)
2. Amount Received - transaction monetary value
3. Received Currency - integer-encoded currency type
4. Payment Format - integer-encoded payment method

In [6]:
edge_features = ['Timestamp', 'Amount Received', 'Received Currency', 'Payment Format']

edge_attr = torch.tensor(df_edges[edge_features].to_numpy()).float()

print(f'Edge feature columns : {edge_features}')
print(f'Edge attr shape (num_transactions x num_features) : {tuple(edge_attr.shape)}')
print(f'Sample row 0 : {edge_attr[0].tolist()}')

Edge feature columns : ['Timestamp', 'Amount Received', 'Received Currency', 'Payment Format']
Edge attr shape (num_transactions x num_features) : (15000000, 4)
Sample row 0 : [0.0, 21.329999923706055, 1.0, 0.0]


6. Build Timestamps and Label Tensors (timestamps, y)

timestamps: float tensor used for temporal train/val/test splits and time-delta features
y: long tensor of ground truth labels (0 = legitimate, 1 = money laundering)

In [7]:
timestamps = torch.FloatTensor(df_edges['Timestamp'].to_numpy())

y = torch.LongTensor(df_edges['Is Laundering'].to_numpy())

print(f'Timestamps tensor shape : {tuple(timestamps.shape)}')
print(f'Labels (y) tensor shape : {tuple(y.shape)}')
print(f'Illicit ratio : {y.sum().item():,} / {len(y):,} = {y.float().mean().item()*100:.3f}%')

Timestamps tensor shape : (15000000,)
Labels (y) tensor shape : (15000000,)
Illicit ratio : 15,342 / 15,000,000 = 0.102%


7. Wrap into a PyTorch Geometric Data Object

Combines x, edge_index, edge_attr, timestamps and y into a single PyG Data object that the GNN will consume during training.

In [8]:
from torch_geometric.data import Data

graph_data = Data(
    x = x,
    edge_index = edge_index,
    edge_attr  = edge_attr,
    y = y
)

graph_data.timestamps = timestamps 

print(f'Nodes (accounts) : {graph_data.num_nodes}')
print(f'Edges (transactions) : {graph_data.num_edges}')
print(f'Node feature dim (placeholder 1.0s) : {graph_data.num_node_features}')
print(f'Edge feature dim (Timestamp, Amount, Currency, Format) : {graph_data.num_edge_features}')
print(f'Timestamps attached : {graph_data.timestamps is not None}')
print(f'Label tensor shape : {tuple(graph_data.y.shape)}')
print()
print(graph_data)

Nodes (accounts) : 2061626
Edges (transactions) : 15000000
Node feature dim (placeholder 1.0s) : 1
Edge feature dim (Timestamp, Amount, Currency, Format) : 4
Timestamps attached : True
Label tensor shape : (15000000,)

Data(x=[2061626, 1], edge_index=[2, 15000000], edge_attr=[15000000, 4], y=[15000000], timestamps=[15000000])


8. Save the graph_data Object

Saves the full PyG Data object to disk so other notebooks can load it directly
without rebuilding all tensors from scratch.

In [9]:
import torch
import os

save_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'model', 'graph_data.pt')
torch.save(graph_data, save_path)

print(f'Saved graph_data -> {save_path}')
print(f'File size: {os.path.getsize(save_path) / 1e9:.2f} GB')

Saved graph_data -> /home/shreyas-nalle/Desktop/Delusional/model/graph_data.pt
File size: 0.67 GB
